In [1]:
import sys
print('Current kernel executable:', sys.executable)

# Install project dependencies into the active notebook kernel
%pip install -r ../requirements.txt

Current kernel executable: c:\repos\adni-project\.venv\Scripts\python.exe
Note: you may need to restart the kernel to use updated packages.


## Environment Setup

Use Python 3.11 specifically for the project venv.

> Run these commands in a terminal (notebook cells cannot activate a new environment for the current kernel).

```powershell
py -3.11 -m venv .venv
.venv\Scripts\activate
python --version
python -m pip install --upgrade pip
pip install -r requirements.txt
```

After selecting the `.venv` kernel in VS Code, run Cell 1 to ensure packages are installed in that kernel.

# ADNI Preprocessing and Sequence Pipeline

This notebook prepares longitudinal ADNI data for LSTM/GRU modeling using the CSV files in `data/`.

Pipeline included:
- Multi-file merge on `PTID` + `VISCODE` (and static merge on `PTID`)
- Categorical encoding, KNN imputation, and MinMax scaling
- 3D sequence tensor generation for recurrent models
- Optional LSTM baselines for both MMSE forecasting and stage prediction

In [6]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [7]:
DATA_DIR = Path('..') / 'data'

FILE_MAP = {
    'ptdemog': 'PTDEMOG_02Apr2026.csv',
    'apoeres': 'APOERES_02Apr2026.csv',
    'mmse': 'MMSE_02Apr2026.csv',
    'adas': 'ADAS_02Apr2026.csv',
    'mri': 'ADNI_PICSLASHS_02Apr2026.csv',
    'cdr': 'CDR_02Apr2026.csv',
    'dx': 'DXSUM_02Apr2026.csv',
}

for key, filename in FILE_MAP.items():
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing file for {key}: {path}')

print('All expected files found.')

All expected files found.


In [8]:
def read_adni_csv(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / filename)
    # ADNI exports sometimes include quoted newlines/spaces in headers.
    df.columns = [c.strip().replace('\n', '').replace('\r', '') for c in df.columns]
    return df

ptdemog = read_adni_csv(FILE_MAP['ptdemog'])
apoeres = read_adni_csv(FILE_MAP['apoeres'])
mmse = read_adni_csv(FILE_MAP['mmse'])
adas = read_adni_csv(FILE_MAP['adas'])
mri = read_adni_csv(FILE_MAP['mri'])
cdr = read_adni_csv(FILE_MAP['cdr'])
dx = read_adni_csv(FILE_MAP['dx'])

for name, df in [('PTDEMOG', ptdemog), ('APOERES', apoeres), ('MMSE', mmse), ('ADAS', adas), ('MRI', mri), ('CDR', cdr), ('DXSUM', dx)]:
    print(f'{name:8s} shape: {df.shape}')

PTDEMOG  shape: (6149, 84)
APOERES  shape: (3208, 16)
MMSE     shape: (14674, 58)
ADAS     shape: (12968, 16)
MRI      shape: (4128, 67)
CDR      shape: (14706, 25)
DXSUM    shape: (16016, 41)


## 1) Build static and longitudinal tables

Static features are attached by `PTID`. Longitudinal features are joined by `PTID` + `VISCODE`.

In [9]:
def normalize_visit_code(vis: pd.Series) -> pd.Series:
    return vis.astype(str).str.strip().str.lower()

def visit_to_month(v: str) -> float:
    if pd.isna(v):
        return np.nan
    v = str(v).strip().lower()
    if v in {'bl', 'm0'}:
        return 0.0
    if v in {'sc', 'scmri', 'init'}:
        return -1.0
    if v.startswith('m') and v[1:].isdigit():
        return float(v[1:])
    return np.nan

for df in [ptdemog, apoeres, mmse, adas, mri, cdr, dx]:
    if 'VISCODE' in df.columns:
        df['VISCODE'] = normalize_visit_code(df['VISCODE'])

ptdemog['VISMONTH'] = ptdemog['VISCODE'].map(visit_to_month)

# Some ADNI exports do not have AGE directly; derive it from visit date and birth year.
if 'AGE' not in ptdemog.columns:
    ptdemog['AGE'] = np.nan

    if 'VISDATE' in ptdemog.columns and 'PTDOBYY' in ptdemog.columns:
        visit_year = pd.to_datetime(ptdemog['VISDATE'], errors='coerce').dt.year
        birth_year = pd.to_numeric(ptdemog['PTDOBYY'], errors='coerce')
        ptdemog['AGE'] = visit_year - birth_year

    # Fallback if VISDATE parsing fails for some rows: infer from PTDOB if possible.
    if ptdemog['AGE'].isna().any() and 'PTDOB' in ptdemog.columns and 'VISDATE' in ptdemog.columns:
        visit_date = pd.to_datetime(ptdemog['VISDATE'], errors='coerce')
        birth_date = pd.to_datetime(ptdemog['PTDOB'], errors='coerce')
        age_years = (visit_date - birth_date).dt.days / 365.25
        ptdemog['AGE'] = ptdemog['AGE'].fillna(age_years)

for col in ['PTGENDER', 'PTEDUCAT']:
    if col not in ptdemog.columns:
        ptdemog[col] = np.nan

demog_cols = ['PTID', 'VISMONTH', 'AGE', 'PTGENDER', 'PTEDUCAT']
demog_base = (
    ptdemog[demog_cols]
    .sort_values(['PTID', 'VISMONTH'])
    .drop_duplicates(subset=['PTID'], keep='first')
)

apoe_col = 'GENOTYPE' if 'GENOTYPE' in apoeres.columns else 'APOE4'
apoe = apoeres[['PTID', apoe_col]].drop_duplicates(subset=['PTID'], keep='last').copy()
apoe = apoe.rename(columns={apoe_col: 'APOE_RAW'})

static_data = demog_base.merge(apoe, on='PTID', how='left')

mmse_long = mmse[['PTID', 'VISCODE', 'MMSCORE']].rename(columns={'MMSCORE': 'MMSE'})
adas_long = adas[['PTID', 'VISCODE', 'TOTAL13']].rename(columns={'TOTAL13': 'ADAS13'})
cdr_long = cdr[['PTID', 'VISCODE', 'CDRSB']].rename(columns={'CDRSB': 'CDRSB'})

mri_cols = ['PTID', 'VISCODE', 'LEFT_HIPP_VOL', 'RIGHT_HIPP_VOL']
mri_long = mri[mri_cols].copy()
mri_long['HIPP_TOTAL'] = mri_long[['LEFT_HIPP_VOL', 'RIGHT_HIPP_VOL']].sum(axis=1, min_count=1)
mri_long = mri_long[['PTID', 'VISCODE', 'HIPP_TOTAL']]

dx_long = dx[['PTID', 'VISCODE', 'DIAGNOSIS', 'DXNORM', 'DXMCI', 'DXAD']].copy()

long_data = mmse_long.merge(adas_long, on=['PTID', 'VISCODE'], how='outer')
long_data = long_data.merge(cdr_long, on=['PTID', 'VISCODE'], how='outer')
long_data = long_data.merge(mri_long, on=['PTID', 'VISCODE'], how='outer')
long_data = long_data.merge(dx_long, on=['PTID', 'VISCODE'], how='left')

master_df = long_data.merge(static_data, on='PTID', how='inner')
master_df = master_df.dropna(subset=['PTID', 'VISCODE']).copy()
master_df['VISMONTH'] = master_df['VISCODE'].map(visit_to_month)

print('static_data:', static_data.shape)
print('long_data  :', long_data.shape)
print('master_df  :', master_df.shape)
master_df.head()

static_data: (4868, 6)
long_data  : (18224, 10)
master_df  : (18162, 15)


,PTID,VISCODE,MMSE,ADAS13,CDRSB,HIPP_TOTAL,DIAGNOSIS,DXNORM,DXMCI,DXAD,VISMONTH,AGE,PTGENDER,PTEDUCAT,APOE_RAW
0,002_S_0295,bl,NaN,4.00,NaN,NaN,1.0,1.0,-4.0,-4.0,0.0,84.84052,1.0,18.0,3/4
1,002_S_0295,m06,28.0,6.33,0.0,NaN,1.0,1.0,-4.0,-4.0,6.0,84.84052,1.0,18.0,3/4
2,002_S_0295,m12,30.0,5.67,0.0,NaN,1.0,1.0,-4.0,-4.0,12.0,84.84052,1.0,18.0,3/4
3,002_S_0295,m24,29.0,5.67,0.0,NaN,1.0,1.0,-4.0,-4.0,24.0,84.84052,1.0,18.0,3/4
4,002_S_0295,m36,28.0,6.67,0.0,NaN,1.0,1.0,-4.0,-4.0,36.0,84.84052,1.0,18.0,3/4


## 2) Cleaning, feature engineering, and imputation

Improvements over a minimal pipeline:
- Robust APOE extraction from genotype strings
- Numeric stage target (`CN`, `MCI`, `AD`) from DXSUM flags
- Leakage-safe split by patient done before scaling/imputation fitting

In [10]:
def parse_apoe_count(val):
    if pd.isna(val):
        return np.nan
    s = str(val).replace('/', '').replace(' ', '')
    digits = [ch for ch in s if ch.isdigit()]
    if len(digits) < 2:
        return np.nan
    return float(sum(1 for d in digits[:2] if d == '4'))

def infer_stage(row):
    # Priority: explicit one-hot diagnosis flags when available.
    if row.get('DXAD', 0) == 1:
        return 'AD'
    if row.get('DXMCI', 0) == 1:
        return 'MCI'
    if row.get('DXNORM', 0) == 1:
        return 'CN'

    diag_val = row.get('DIAGNOSIS', np.nan)
    if pd.notna(diag_val):
        try:
            d = int(diag_val)
            if d == 1:
                return 'CN'
            if d == 2:
                return 'MCI'
            if d == 3:
                return 'AD'
        except Exception:
            pass

    return np.nan

master_df['PTGENDER'] = master_df['PTGENDER'].astype(str).str.strip().str.lower().map({'male': 0.0, 'female': 1.0})
master_df['APOE4'] = master_df['APOE_RAW'].apply(parse_apoe_count)
master_df['STAGE'] = master_df.apply(infer_stage, axis=1)
master_df['STAGE_CODE'] = master_df['STAGE'].map({'CN': 0, 'MCI': 1, 'AD': 2})

master_df = master_df.sort_values(['PTID', 'VISMONTH', 'VISCODE'])
master_df = master_df.drop_duplicates(subset=['PTID', 'VISCODE'], keep='last').copy()

display(master_df[['PTID', 'VISCODE', 'MMSE', 'ADAS13', 'CDRSB', 'HIPP_TOTAL', 'PTGENDER', 'AGE', 'PTEDUCAT', 'APOE4', 'STAGE']].head(10))
print('Unique patients:', master_df['PTID'].nunique())
print('Unique visits  :', master_df[['PTID', 'VISCODE']].drop_duplicates().shape[0])
print('Stage counts   :')
print(master_df['STAGE'].value_counts(dropna=False))

,PTID,VISCODE,MMSE,ADAS13,CDRSB,HIPP_TOTAL,PTGENDER,AGE,PTEDUCAT,APOE4,STAGE
6,002_S_0295,sc,28.0,NaN,0.0,NaN,NaN,84.840520,18.0,1.0,NaN
0,002_S_0295,bl,NaN,4.00,NaN,NaN,NaN,84.840520,18.0,1.0,CN
1,002_S_0295,m06,28.0,6.33,0.0,NaN,NaN,84.840520,18.0,1.0,CN
2,002_S_0295,m12,30.0,5.67,0.0,NaN,NaN,84.840520,18.0,1.0,CN
3,002_S_0295,m24,29.0,5.67,0.0,NaN,NaN,84.840520,18.0,1.0,CN
4,002_S_0295,m36,28.0,6.67,0.0,NaN,NaN,84.840520,18.0,1.0,CN
5,002_S_0295,m48,26.0,8.00,0.0,NaN,NaN,84.840520,18.0,1.0,CN
7,002_S_0295,v06,28.0,9.33,0.0,NaN,NaN,84.840520,18.0,1.0,CN
8,002_S_0295,v11,22.0,9.33,0.0,NaN,NaN,84.840520,18.0,1.0,CN
11,002_S_0413,init,30.0,4.33,0.0,4086.906,NaN,76.344969,16.0,0.0,CN


Unique patients: 4624
Unique visits  : 18159
Stage counts   :
STAGE
MCI    6536
CN     6120
AD     2996
NaN    2507
Name: count, dtype: int64


In [14]:
feature_cols = ['MMSE', 'ADAS13', 'CDRSB', 'HIPP_TOTAL', 'AGE', 'PTGENDER', 'PTEDUCAT', 'APOE4']
required_for_sequence = ['PTID', 'VISCODE', 'VISMONTH'] + feature_cols

work_df = master_df[required_for_sequence + ['STAGE_CODE']].copy()

# Keep rows that are at least locatable in time and patient index.
work_df = work_df.dropna(subset=['PTID', 'VISCODE', 'VISMONTH'])

patient_ids = work_df['PTID'].dropna().unique()
train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)

train_df = work_df[work_df['PTID'].isin(train_ids)].copy()
test_df = work_df[work_df['PTID'].isin(test_ids)].copy()

# Force numeric for imputation inputs; non-numeric values become NaN.
train_num = train_df[feature_cols].apply(pd.to_numeric, errors='coerce')
test_num = test_df[feature_cols].apply(pd.to_numeric, errors='coerce')

try:
    imputer = KNNImputer(n_neighbors=5, keep_empty_features=True)
except TypeError:
    imputer = KNNImputer(n_neighbors=5)

scaler = MinMaxScaler()

train_imp = imputer.fit_transform(train_num)
test_imp = imputer.transform(test_num)

# Some sklearn versions can drop all-NaN columns; rebuild to the original feature set safely.
imp_cols = list(getattr(imputer, 'feature_names_in_', feature_cols))
train_imp_df = pd.DataFrame(train_imp, index=train_df.index, columns=imp_cols)
test_imp_df = pd.DataFrame(test_imp, index=test_df.index, columns=imp_cols)

for col in feature_cols:
    if col not in train_imp_df.columns:
        train_imp_df[col] = 0.0
    if col not in test_imp_df.columns:
        test_imp_df[col] = 0.0

train_imp_df = train_imp_df[feature_cols]
test_imp_df = test_imp_df[feature_cols]

train_scaled = scaler.fit_transform(train_imp_df)
test_scaled = scaler.transform(test_imp_df)

train_df.loc[:, feature_cols] = train_scaled
test_df.loc[:, feature_cols] = test_scaled

# Optional for classification target continuity; forward fill stage per patient if missing.
train_df['STAGE_CODE'] = train_df.groupby('PTID')['STAGE_CODE'].ffill().bfill()
test_df['STAGE_CODE'] = test_df.groupby('PTID')['STAGE_CODE'].ffill().bfill()

print('Train rows:', train_df.shape[0], '| Test rows:', test_df.shape[0])
print('Train patients:', train_df['PTID'].nunique(), '| Test patients:', test_df['PTID'].nunique())
print('Columns used for features:', feature_cols)
train_df.head()

Train rows: 6247 | Test rows: 1473
Train patients: 2025 | Test patients: 507
Columns used for features: ['MMSE', 'ADAS13', 'CDRSB', 'HIPP_TOTAL', 'AGE', 'PTGENDER', 'PTEDUCAT', 'APOE4']


,PTID,VISCODE,VISMONTH,MMSE,ADAS13,CDRSB,HIPP_TOTAL,AGE,PTGENDER,PTEDUCAT,APOE4,STAGE_CODE
6,002_S_0295,sc,-1.0,0.935484,0.200908,0.000000,0.150488,0.851016,0.0,0.875,0.5,0.0
0,002_S_0295,bl,0.0,0.858065,0.055043,0.088889,0.167865,0.851016,0.0,0.875,0.5,0.0
1,002_S_0295,m06,6.0,0.935484,0.087106,0.000000,0.162468,0.851016,0.0,0.875,0.5,0.0
2,002_S_0295,m12,12.0,1.000000,0.078024,0.000000,0.154492,0.851016,0.0,0.875,0.5,0.0
3,002_S_0295,m24,24.0,0.967742,0.078024,0.000000,0.154492,0.851016,0.0,0.875,0.5,0.0


## 3) Sequence generation for RNNs

Creates tensors with shape `(samples, time_steps, features)`.

- `y_mmse`: next-step MMSE (regression)
- `y_stage`: next-step stage code (`0=CN, 1=MCI, 2=AD`)

In [15]:
def create_sequences(df, features, sequence_length=3, target_col_mmse='MMSE', target_col_stage='STAGE_CODE'):
    X, y_mmse, y_stage, patient_seq = [], [], [], []

    for ptid, group in df.groupby('PTID'):
        group = group.sort_values(['VISMONTH', 'VISCODE'])
        vals = group[features].values
        mmse_vals = group[target_col_mmse].values
        stage_vals = group[target_col_stage].values

        if len(vals) >= sequence_length + 1:
            for i in range(len(vals) - sequence_length):
                X.append(vals[i:i + sequence_length])
                y_mmse.append(mmse_vals[i + sequence_length])
                y_stage.append(stage_vals[i + sequence_length])
                patient_seq.append(ptid)

    return np.array(X), np.array(y_mmse), np.array(y_stage), np.array(patient_seq)

SEQUENCE_LENGTH = 3

X_train, y_train_mmse, y_train_stage, p_train = create_sequences(train_df, feature_cols, sequence_length=SEQUENCE_LENGTH)
X_test, y_test_mmse, y_test_stage, p_test = create_sequences(test_df, feature_cols, sequence_length=SEQUENCE_LENGTH)

print('Train X shape:', X_train.shape)
print('Train y_mmse shape:', y_train_mmse.shape)
print('Train y_stage shape:', y_train_stage.shape)
print('Test X shape:', X_test.shape)
print('Test y_mmse shape:', y_test_mmse.shape)
print('Test y_stage shape:', y_test_stage.shape)

if X_train.size > 0:
    print('Example sequence tensor shape:', X_train[0].shape)

Train X shape: (2140, 3, 8)
Train y_mmse shape: (2140,)
Train y_stage shape: (2140,)
Test X shape: (496, 3, 8)
Test y_mmse shape: (496,)
Test y_stage shape: (496,)
Example sequence tensor shape: (3, 8)


## 4) Optional: LSTM baselines

Run these cells after verifying sequence shapes.

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report

tf.random.set_seed(42)
np.random.seed(42)

def build_lstm_regressor(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

def build_lstm_classifier(input_shape, n_classes=3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [17]:
if X_train.size == 0 or X_test.size == 0:
    print('Not enough longitudinal visits to create train/test sequences with current settings.')
else:
    es = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

    reg_model = build_lstm_regressor((X_train.shape[1], X_train.shape[2]))
    reg_model.fit(
        X_train, y_train_mmse,
        validation_split=0.2,
        epochs=60,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    y_pred_mmse = reg_model.predict(X_test, verbose=0).ravel()
    print('MMSE MAE:', round(mean_absolute_error(y_test_mmse, y_pred_mmse), 4))

    cls_model = build_lstm_classifier((X_train.shape[1], X_train.shape[2]), n_classes=3)
    cls_model.fit(
        X_train, y_train_stage,
        validation_split=0.2,
        epochs=60,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    y_pred_stage = np.argmax(cls_model.predict(X_test, verbose=0), axis=1)
    print('Stage Accuracy:', round(accuracy_score(y_test_stage, y_pred_stage), 4))
    print(classification_report(y_test_stage, y_pred_stage, target_names=['CN', 'MCI', 'AD'], zero_division=0))

MMSE MAE: 0.0631
Stage Accuracy: 0.619
              precision    recall  f1-score   support

          CN       0.81      0.67      0.74       137
         MCI       0.91      0.30      0.46       200
          AD       0.49      0.97      0.65       159

    accuracy                           0.62       496
   macro avg       0.74      0.65      0.61       496
weighted avg       0.75      0.62      0.60       496



## 5) Export processed artifacts

Saves cleaned tabular data and sequence tensors for reuse in training scripts.

In [18]:
OUT_DIR = Path('..') / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(OUT_DIR / 'train_longitudinal_clean.csv', index=False)
test_df.to_csv(OUT_DIR / 'test_longitudinal_clean.csv', index=False)

np.save(OUT_DIR / 'X_train.npy', X_train)
np.save(OUT_DIR / 'X_test.npy', X_test)
np.save(OUT_DIR / 'y_train_mmse.npy', y_train_mmse)
np.save(OUT_DIR / 'y_test_mmse.npy', y_test_mmse)
np.save(OUT_DIR / 'y_train_stage.npy', y_train_stage)
np.save(OUT_DIR / 'y_test_stage.npy', y_test_stage)

print('Saved artifacts to:', OUT_DIR.resolve())

Saved artifacts to: C:\repos\adni-project\outputs
